# Mask R-CNN Instance Segmentation — Dog & Cat Dataset

This notebook demonstrates training a **Mask R-CNN** model using the [Matterport Mask-RCNN](https://github.com/matterport/Mask_RCNN) implementation on a custom **Dog & Cat** dataset (COCO format).

**Advanced MLOps Pipeline Features:**
1. **Full COCO Dataset Parsing Engine** (Loads Images + Polygon Masks properly)
2. **Data Augmentation** (imgaug integration: Flips, Blurs) to improve accuracy
3. Configure and train Mask R-CNN
4. Run inference & evaluation metrics

> **Prerequisites:** Set your Roboflow API key as an environment variable:  
> `export ROBOFLOW_API_KEY="your_key_here"`

In [ ]:
# ── Step 1: Install Dependencies ─────────────────────────────────────────────
!pip install git+https://github.com/matterport/Mask_RCNN.git --quiet
!pip install roboflow scikit-learn Pillow pycocotools imgaug --quiet

In [ ]:
# ── Step 2: Download Dataset from Roboflow (COCO format) ─────────────────────
import os
from roboflow import Roboflow

api_key = os.environ.get("ROBOFLOW_API_KEY", "")
if not api_key:
    raise EnvironmentError("ROBOFLOW_API_KEY environment variable is not set.")

rf = Roboflow(api_key=api_key)
project = rf.workspace("appledetect").project("dog_cat_objects")
version = project.version(1)
dataset = version.download("coco")  # Download in COCO format

print(f"Dataset downloaded to: {dataset.location}")

In [ ]:
# ── Step 3: Implement COCO Dataset Parser ────────────────────────────────────
import json
import numpy as np
import skimage.draw
from mrcnn import utils

class CustomCocoDataset(utils.Dataset):
    def load_custom_coco(self, dataset_dir, subset):
        """Load a subset of the COCO dataset."""
        # Add classes (1 for cats, 2 for dogs)
        self.add_class("dataset", 1, "cats")
        self.add_class("dataset", 2, "dogs")
        
        # Load JSON annotations
        json_path = os.path.join(dataset_dir, subset, "_annotations.coco.json")
        with open(json_path) as f:
            coco_data = json.load(f)
            
        # Load images and corresponding annotations
        for img in coco_data['images']:
            annots = [a for a in coco_data['annotations'] if a['image_id'] == img['id']]
            
            self.add_image(
                "dataset",
                image_id=img['id'],
                path=os.path.join(dataset_dir, subset, img['file_name']),
                width=img['width'],
                height=img['height'],
                annotations=annots
            )

    def load_mask(self, image_id):
        """Generate instance masks for an image."""
        image_info = self.image_info[image_id]
        annotations = image_info["annotations"]
        
        mask = np.zeros([image_info["height"], image_info["width"], len(annotations)], dtype=np.uint8)
        class_ids = []
        
        for i, ann in enumerate(annotations):
            # Convert COCO polygon segmentation to binary mask
            poly = np.array(ann['segmentation'][0]).reshape((int(len(ann['segmentation'][0])/2), 2))
            rr, cc = skimage.draw.polygon(poly[:, 1], poly[:, 0], mask.shape[:2])
            mask[rr, cc, i] = 1
            class_ids.append(ann['category_id'])
            
        # Handle case where there are no annotations
        if len(class_ids) == 0:
            mask = np.empty([image_info["height"], image_info["width"], 0], dtype=np.uint8)
            class_ids = np.empty([0], dtype=np.int32)
            
        return mask.astype(np.bool), np.array(class_ids, dtype=np.int32)

# Initialize and prepare training & validation datasets
train_dataset = CustomCocoDataset()
train_dataset.load_custom_coco(dataset.location, "train")
train_dataset.prepare()

val_dataset = CustomCocoDataset()
val_dataset.load_custom_coco(dataset.location, "valid")
val_dataset.prepare()

print(f"Training images: {len(train_dataset.image_ids)}")
print(f"Validation images: {len(val_dataset.image_ids)}")

In [ ]:
# ── Step 4: Configure and Train Mask R-CNN ───────────────────────────────────
import mrcnn
from mrcnn import model as modellib
from mrcnn.config import Config
from imgaug import augmenters as iaa

MODEL_DIR = os.path.join(os.getcwd(), "mrcnn_logs")
os.makedirs(MODEL_DIR, exist_ok=True)

class DogCatConfig(Config):
    NAME = "dog_cat"
    IMAGES_PER_GPU = 1       
    NUM_CLASSES = 1 + 2      # Background + cats + dogs
    STEPS_PER_EPOCH = 50     # Adjust based on dataset size
    DETECTION_MIN_CONFIDENCE = 0.9

config = DogCatConfig()
model = modellib.MaskRCNN(mode="training", config=config, model_dir=MODEL_DIR)

# Transfer Learning: Load COCO weights (excluding output layers)
COCO_WEIGHTS_PATH = "mask_rcnn_coco.h5"
if not os.path.exists(COCO_WEIGHTS_PATH):
    utils.download_trained_weights(COCO_WEIGHTS_PATH)
    
model.load_weights(COCO_WEIGHTS_PATH, by_name=True, exclude=[
    "mrcnn_class_logits", "mrcnn_bbox_fc",
    "mrcnn_bbox", "mrcnn_mask"
])

# Advanced MLOps: Apply Data Augmentation (Flip and Gaussian Blur)
augmentation = iaa.Sometimes(0.5, [
    iaa.Fliplr(0.5),
    iaa.GaussianBlur(sigma=(0.0, 1.0))
])

# Train the network (heads only for fine-tuning)
print("Starting training with Image Augmentation...")
model.train(
    train_dataset, val_dataset,
    learning_rate=config.LEARNING_RATE,
    epochs=5,
    layers="heads",
    augmentation=augmentation
)

In [ ]:
# ── Step 5: Run Inference on a Test Image ────────────────────────────────────
import skimage.io
from mrcnn import visualize
import random

CLASS_NAMES = ["BG", "cats", "dogs"] 

# Switch model to inference mode
inference_config = DogCatConfig()
inference_model = modellib.MaskRCNN(
    mode="inference", config=inference_config, model_dir=MODEL_DIR
)

# Load the best/last saved weights
inference_model.load_weights(inference_model.find_last(), by_name=True)

# Run prediction on a random validation image
image_id = random.choice(val_dataset.image_ids)
image = val_dataset.load_image(image_id)

results = inference_model.detect([image], verbose=0)
r = results[0]

visualize.display_instances(
    image, r["rois"], r["masks"], r["class_ids"],
    CLASS_NAMES, r["scores"]
)